### Упражнение 2. Атомарность при ошибке

**Задание.** Смоделируйте ситуацию, где часть строк вставляется успешно, а затем происходит ошибка, и убедитесь, что откатилось всё.

**Шаги.**
1. В отдельной транзакции создайте таблицу `ex2 (id INT PRIMARY KEY, name TEXT)` — без дубликатов.
2. В следующей транзакции выполните `executemany` со списком `[(1,'a'), (2,'b'), (1,'dup')]`.
3. Оберните в `try/except UniqueViolation`.
4. Проверьте `SELECT count(*) FROM ex2` — должно быть 0 (откатились даже успешные вставки).

**Критерий успеха.** Счётчик 0, в консоли — сообщение об ошибке.


In [22]:
# УПРАЖНЕНИЕ 2
# Транзакции: «всё или ничего». Если в пачке одна строка битая, откатывается ВСЯ пачка
# (как кнопка «Отменить» в Word, только для целой группы действий).

from psycopg.errors import UniqueViolation

# --- Шаг 1. Создаём таблицу в ОТДЕЛЬНОЙ транзакции и фиксируем её ---
# Если бы CREATE TABLE был в той же транзакции, что и ошибочная вставка,
# при откате исчезла бы и таблица (DDL в PostgreSQL транзакционен),
# и SELECT count(*) упал бы с ошибкой вместо того, чтобы вернуть 0.
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS ex2")
        cur.execute("CREATE TABLE ex2 (id INT PRIMARY KEY, name TEXT)")
    conn.commit()   # таблица зафиксирована, откат её уже не заденет

# --- Шаг 2-3. Все три вставки — в ОДНОЙ транзакции; try стоит СНАРУЖИ with ---
try:
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            # (1,'a') и (2,'b') вставятся, а (1,'dup') нарушит PRIMARY KEY -> UniqueViolation
            cur.executemany(
                "INSERT INTO ex2 VALUES (%s, %s)",
                [(1, "a"), (2, "b"), (1, "dup")],
            )
        conn.commit()   # сюда выполнение не дойдёт: исключение вылетит раньше
except UniqueViolation as e:
    # Исключение «пролетело» через блок with -> psycopg сам сделал ROLLBACK и закрыл соединение
    print("Поймали UniqueViolation:", str(e).splitlines()[0])

# --- Шаг 4. Проверяем НОВЫМ соединением ---
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT count(*) FROM ex2")
        print("Строк в таблице:", cur.fetchone()[0])   # 0 — откатились даже «хорошие» (1,'a') и (2,'b')


# --- БОНУС: если часть данных ЗАФИКСИРОВАТЬ до ошибки, она останется ---
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.executemany("INSERT INTO ex2 VALUES (%s, %s)", [(1, "a"), (2, "b")])
    conn.commit()                                   # транзакция №1 зафиксирована

try:
    with psycopg.connect(PSYCOPG_CONN) as conn:     # транзакция №2 — отдельная
        with conn.cursor() as cur:
            cur.execute("INSERT INTO ex2 VALUES (%s, %s)", (1, "dup"))   # дубль -> ошибка
except UniqueViolation:
    print("транзакция №2 откатилась")

with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT count(*) FROM ex2")
        print("Строк после бонуса:", cur.fetchone()[0])   # 2 — откатывается только транзакция с ошибкой

Поймали UniqueViolation: duplicate key value violates unique constraint "ex2_pkey"
Строк в таблице: 0
транзакция №2 откатилась
Строк после бонуса: 2


### Упражнение 3. Почините уязвимый запрос

**Задание.** Дана функция `broken(code)` — она ищет пилота по коду через f-string. Перепишите её безопасно.

**Шаги.**
1. Скопируйте функцию, замените f-string на `%s` + параметр.
2. Проверьте: при `code = "' OR '1'='1"` результат должен быть пустым.
3. Дополнительно: добавьте `LIMIT 5` в SQL.

**Критерий успеха.** `broken("' OR '1'='1")` (старая версия) вернёт всё, `fixed("' OR '1'='1")` — пусто.


In [13]:
# УПРАЖНЕНИЕ 3
# SQL-инъекции: Главная уязвимость хакеров. 
# Если ты подставляешь данные пользователя прямо в строку SQL, хакер может удалить базу. Решение — параметризация.

def broken(code):
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT id, name FROM demo_items WHERE name = '{code}'")
            return cur.fetchall()

def fixed(code):
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT id, name FROM demo_items WHERE name = %s LIMIT 5", (code,)) # ПРОВЕРЯЕТ РАВНО ЛИ ИМЯ ("' OR '1'='1' --" # пример хакерской атаки)
            return cur.fetchall()

# TODO: напишите безопасную версию fixed(code)
# TODO: проверьте на "' OR '1'='1"

code = "' OR '1'='1' --" # пример хакерской атаки
print("unsafe :", broken(code))
print("safe   :", fixed(code))


unsafe : [(1, 'apple'), (2, 'banana')]
safe   : []


### Упражнение 4. Батч на 10 000 строк

**Задание.** Сравните цикл и `executemany` на 10 000 строк и найдите порог, на котором `executemany` стабильно быстрее.

**Шаги.**
1. Сгенерируйте 10 000 кортежей `(id, name, val)`.
2. Замерьте цикл, замерьте `executemany`.
3. Напечатайте `speedup = t_loop / t_batch`.
4. **Вопрос на подумать:** почему `executemany` не «в N раз быстрее» линейно?

**Критерий успеха.** `speedup > 5`. В комментарии — 1–2 строки объяснения.


In [23]:
# УПРАЖНЕНИЕ 4
# Цель: увидеть, что executemany (пачка) быстрее цикла из execute, и найти порог выигрыша.

import time, random
import psycopg

# --- Таблица для замеров ---
with psycopg.connect(PSYCOPG_CONN) as conn:
    with conn.cursor() as cur:
        cur.execute("DROP TABLE IF EXISTS bench10k")
        cur.execute("CREATE TABLE bench10k (id INT PRIMARY KEY, name TEXT, val INT)")
    conn.commit()

def make_rows(n):
    """n кортежей (id, name, val)."""
    return [(i, f"item_{i}", random.randint(1, 100)) for i in range(1, n + 1)]

def truncate():
    """Очищаем таблицу ВНЕ замера (иначе будет конфликт id и лишнее время)."""
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            cur.execute("TRUNCATE bench10k")
        conn.commit()

def time_loop(rows):
    """Цикл: один execute на строку -> N обращений к серверу."""
    truncate()
    t0 = time.perf_counter()                       # включили секундомер
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            for r in rows:
                cur.execute("INSERT INTO bench10k VALUES (%s, %s, %s)", r)
        conn.commit()
    return time.perf_counter() - t0                # сколько секунд прошло

def time_batch(rows):
    """Пачка: один вызов executemany на все строки."""
    truncate()
    t0 = time.perf_counter()
    with psycopg.connect(PSYCOPG_CONN) as conn:
        with conn.cursor() as cur:
            cur.executemany("INSERT INTO bench10k VALUES (%s, %s, %s)", rows)
        conn.commit()
    return time.perf_counter() - t0

# --- Ищем порог: с какого размера батч заметно выигрывает ---
print(f"{'N':>7} | {'цикл, c':>8} | {'batch, c':>8} | ускорение")
for n in (10, 100, 1000, 10_000):
    rows = make_rows(n)
    t_loop, t_batch = time_loop(rows), time_batch(rows)
    print(f"{n:>7} | {t_loop:>8.3f} | {t_batch:>8.3f} | x{t_loop / max(t_batch, 1e-9):.1f}")

# --- Основной результат: 10 000 строк ---
rows = make_rows(10_000)
t_loop, t_batch = time_loop(rows), time_batch(rows)
speedup = t_loop / max(t_batch, 1e-9)              # max(...) защищает от деления на ноль
print(f"\nЦикл: {t_loop:.2f}s, Батч: {t_batch:.2f}s")
print(f"Ускорение: x{speedup:.1f}   (> 5 ? {speedup > 5})")

# ПОЧЕМУ НЕ «В 10 000 РАЗ БЫСТРЕЕ»:
# executemany экономит только СЕТЕВЫЕ ОБРАЩЕНИЯ (пинг-понг клиент <-> сервер).
# Всё остальное от способа вставки не зависит: сервер всё равно разбирает и выполняет вставку
# каждой строки, обновляет индекс PRIMARY KEY, пишет WAL и делает commit.
# К тому же сервер на localhost (сетевая задержка почти нулевая), так что по настоящей сети
# выигрыш был бы больше. Для миллионов строк быстрее всего COPY (cur.copy()).

      N |  цикл, c | batch, c | ускорение
     10 |    0.008 |    0.005 | x1.6
    100 |    0.010 |    0.006 | x1.7
   1000 |    0.081 |    0.012 | x6.8
  10000 |    0.774 |    0.080 | x9.7

Цикл: 0.78s, Батч: 0.08s
Ускорение: x9.5   (> 5 ? True)


### Упражнение 5. Core: создать таблицу и залить батч

**Задание.** Через `text()` и `engine.connect()` создайте таблицу `ex5_measurements (id, sensor, value)` и вставьте 200 строк списком словарей.

**Шаги.**
1. `DROP TABLE IF EXISTS ex5_measurements` и `CREATE TABLE ...`.
2. Сгенерируйте `records` — список словарей `{"id", "sensor", "value"}` на 200 элементов.
3. Один вызов `conn.execute(text("INSERT ..."), records)`.
4. Проверьте: `SELECT count(*) FROM ex5_measurements WHERE value > 50`.

**Критерий успеха.** В консоли: `count > 0`. В логах — один `INSERT` (executemany), не 200.


In [24]:
# УПРАЖНЕНИЕ 5
# SQLAlchemy Core: если вторым аргументом execute передать СПИСОК СЛОВАРЕЙ,
# Core сам выполнит executemany (одна пачка вместо 200 отдельных запросов).

from sqlalchemy import text

with engine.connect() as conn:
    # 1. DDL: пересоздаём таблицу
    conn.execute(text("DROP TABLE IF EXISTS ex5_measurements"))
    conn.execute(text("CREATE TABLE ex5_measurements (id INT PRIMARY KEY, sensor TEXT, value NUMERIC)"))
    conn.commit()

    # 2. 200 записей. Ключи словаря ("id","sensor","value") должны совпадать с именами :id, :sensor, :value
    records = [{"id": i, "sensor": f"S{i % 4}", "value": i * 0.7} for i in range(1, 201)]   # датчики S0..S3

    # 3. ОДИН вызов: список словарей -> executemany
    conn.execute(
        text("INSERT INTO ex5_measurements (id, sensor, value) VALUES (:id, :sensor, :value)"),
        records,
    )
    conn.commit()

    # 4. Проверки
    total = conn.execute(text("SELECT count(*) FROM ex5_measurements")).scalar()
    print("всего строк:", total)                                  # 200
    cnt = conn.execute(
        text("SELECT count(*) FROM ex5_measurements WHERE value > :t"),   # :t — именованный параметр
        {"t": 50},
    ).scalar()                                                     # .scalar() = первое значение первой строки
    print("count > 50:", cnt)                                      # 129 (0.7*i > 50 при i >= 72)

всего строк: 200
count > 50: 129


### Упражнение 6. Агрегат по сенсорам

**Задание.** Читая `ex5_measurements` чанками по 50, посчитайте среднее `value` и количество замеров по каждому `sensor`.

**Шаги.**
1. Итерируйтесь `pd.read_sql(..., engine, chunksize=50)`.
2. Для каждого чанка считайте `groupby("sensor").agg(mean=("value","mean"), n=("value","size"))`.
3. Соберите результаты в словарь `{sensor: {"sum": ..., "n": ...}}` (агрегируйте среднее, пересчитывая взвешенно).
4. Напечатайте итоговую таблицу.

**Критерий успеха.** Сумма `n` по всем сенсорам == общее число строк в `ex5_measurements`.


In [26]:
# УПРАЖНЕНИЕ 6
# Читаем таблицу КУСКАМИ по 50 строк и копим промежуточные итоги.
# Ловушка: нельзя усреднять средние кусков (куски могут быть разного размера).
# Правильно копить СУММУ и КОЛИЧЕСТВО, а среднее считать в самом конце: сумма / количество.

import pandas as pd

acc = {}   # итог: {sensor: {"sum": накопленная сумма, "n": накопленное количество}}

# value::float8 — приводим NUMERIC к обычному числу на стороне SQL,
# чтобы pandas получил числовую колонку, а не объекты Decimal
sql = "SELECT sensor, value::float8 AS value FROM ex5_measurements ORDER BY id"

# chunksize=50 -> read_sql возвращает ИТЕРАТОР: в памяти одновременно не больше 50 строк
for chunk in pd.read_sql(sql, engine, chunksize=50):
    # Группируем КУСОК по сенсору: сумма и количество внутри этого куска
    part = chunk.groupby("sensor").agg(s=("value", "sum"), n=("value", "size"))
    for sensor, row in part.iterrows():
        d = acc.setdefault(sensor, {"sum": 0.0, "n": 0})   # если сенсора ещё нет — заводим нули
        d["sum"] += float(row["s"])                         # прибавляем сумму куска
        d["n"]   += int(row["n"])                           # прибавляем количество куска

# Итоговая таблица: (sensor, mean, count)
result = pd.DataFrame(
    [(s, a["sum"] / a["n"], a["n"]) for s, a in sorted(acc.items())],
    columns=["sensor", "mean", "n"],
)
print(result.to_string(index=False))

# Критерий успеха: сумма n по всем сенсорам == числу строк в таблице
with engine.connect() as conn:
    rows_in_table = conn.execute(text("SELECT count(*) FROM ex5_measurements")).scalar()
print("\nсумма n =", result["n"].sum(), "| строк в таблице =", rows_in_table,
      "| совпало:", result["n"].sum() == rows_in_table)       # 200 == 200 -> True

sensor  mean  n
    S0  71.4 50
    S1  69.3 50
    S2  70.0 50
    S3  70.7 50

сумма n = 200 | строк в таблице = 200 | совпало: True


### Упражнение 7. Своя M:N-модель

**Задание.** Опишите M:N между `Author` и `Book` через ассоциативную `author_book`.

**Шаги.**
1. Объявите `Author` (id, name) и `Book` (id, title) и `author_book` через `Table`.
2. В `relationship` пропишите `secondary=author_book` и `back_populates` с обеих сторон.
3. `create_all`.
4. Добавьте 2 авторов и 3 книги (некоторые — с двумя авторами).
5. Удалите одного автора и проверьте, что связи в `author_book` для него исчезли, а книги, у которых были другие авторы, остались.

**Критерий успеха.** До удаления связей 3–4, после удаления — меньше на число связей удалённого автора.


In [27]:
# УПРАЖНЕНИЕ 7
# M:N («многие ко многим»): у автора много книг, у книги может быть несколько авторов.
# В БД это три таблицы: authors, books и связующая author_book.

from sqlalchemy import ForeignKey, String, Table, Column, select, delete, func
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session

# 1. Свой базовый класс 
class Base2(DeclarativeBase):
    pass

# 2. Связующая таблица. ondelete="CASCADE" — правило ВНУТРИ БД:
#    удалили автора (или книгу) -> строки связи удаляются сами.
#    Составной первичный ключ (author_id, book_id) запрещает дубль пары «автор–книга».
author_book = Table(
    "author_book", Base2.metadata,
    Column("author_id", ForeignKey("authors.id", ondelete="CASCADE"), primary_key=True),
    Column("book_id",   ForeignKey("books.id",   ondelete="CASCADE"), primary_key=True),
)

# 3. Автор. secondary= через какую таблицу связаны; back_populates= имя «обратной» стороны
class Author(Base2):
    __tablename__ = "authors"
    id:   Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    books: Mapped[list["Book"]] = relationship(secondary=author_book, back_populates="authors")

# 4. Книга
class Book(Base2):
    __tablename__ = "books"
    id:    Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(String(200))
    authors: Mapped[list["Author"]] = relationship(secondary=author_book, back_populates="books")

# 5. Чистый старт: drop_all, чтобы при повторном запуске числа не «плыли»
Base2.metadata.drop_all(engine)
Base2.metadata.create_all(engine)

def count_links(s):
    """Сколько строк сейчас в связующей таблице."""
    return s.execute(select(func.count()).select_from(author_book)).scalar()

def count_books(s):
    return s.execute(select(func.count()).select_from(Book)).scalar()

with Session(engine) as s:
    # ВАЖНО: каждую книгу создаём ОДИН раз и отдаём ОДИН И ТОТ ЖЕ объект двум авторам.
    # Если написать Book(title="Война и мир") дважды, получатся две разные строки в books,
    # и общей книги не будет.
    war   = Book(title="Война и мир")      # общая книга
    anna  = Book(title="Анна Каренина")
    idiot = Book(title="Идиот")

    tolstoy = Author(name="Толстой",     books=[war, anna])     # 2 связи
    dosto   = Author(name="Достоевский", books=[war, idiot])    # 2 связи
    s.add_all([tolstoy, dosto])          # книги сохранятся автоматически через связи
    s.commit()

    print("книг:", count_books(s), "| связей ДО удаления:", count_links(s))   # 3 и 4

    # 6. Удаляем Толстого
    s.delete(tolstoy)
    s.commit()

    print("связей ПОСЛЕ удаления Толстого:", count_links(s))   # 2 (4 минус его 2 связи)
    print("книг осталось:", count_books(s))                    # 3 — сами книги НЕ удаляются

    # Проверяем общую книгу: у «Войны и мира» остался один автор — Достоевский
    war_authors = s.scalars(
        select(Author.name).join(author_book).join(Book).where(Book.title == "Война и мир")
    ).all()
    print("авторы «Войны и мира» теперь:", war_authors)         # ['Достоевский']

    # БОНУС: «массовое» удаление мимо ORM — здесь работает именно CASCADE на уровне БД
    s.execute(delete(Author).where(Author.name == "Достоевский"))
    s.commit()
    print("связей после массового удаления:", count_links(s))  # 0

# Заметка: при обычном s.delete(obj) строки связующей таблицы убирает сам ORM.
# ondelete="CASCADE" страхует случаи, когда удаляют напрямую SQL-ом (как в бонусе).

книг: 3 | связей ДО удаления: 4
связей ПОСЛЕ удаления Толстого: 2
книг осталось: 3
авторы «Войны и мира» теперь: ['Достоевский']
связей после массового удаления: 0


### Упражнение 8. `joinedload` и `.unique()`

**Задание.** Решите ту же задачу через `joinedload` и убедитесь, что запрос действительно один.

**Шаги.**
1. Импортируйте `joinedload` из `sqlalchemy.orm`.
2. Замените `selectinload(Pilot.aircrafts)` на `joinedload(Pilot.aircrafts)`.
3. Запустите `eager_load()` — получите `InvalidRequestError` (нужна дедупликация).
4. Исправьте: `s.scalars(stmt).unique().all()`.
5. Пересчитайте запросы через `count_queries`.

**Критерий успеха.** 1 запрос вместо 2. В комментарии — почему `selectinload` всё равно рекомендуют чаще.


In [28]:
# УПРАЖНЕНИЕ 8
# N+1: 1 запрос за родителями + N запросов за детьми каждого.
# Лечим: selectinload (2 запроса) или joinedload (1 запрос с JOIN, но нужен .unique()).

from sqlalchemy.orm import joinedload
from sqlalchemy.exc import InvalidRequestError

# --- Шаг 3 задания: joinedload БЕЗ .unique() -> ошибка ---
def eager_load_joined_wrong():
    with Session(engine) as s:
        stmt = select(Pilot).options(joinedload(Pilot.aircrafts))
        return len(s.scalars(stmt).all())          # здесь будет InvalidRequestError

try:
    count_queries(eager_load_joined_wrong)
except InvalidRequestError as e:
    # JOIN размножает пилота по числу его самолётов (2 строки на пилота),
    # и SQLAlchemy требует явно «схлопнуть» дубли
    print("Без unique():", str(e).splitlines()[0][:100], "...")

# --- Шаг 4: исправленная версия ---
def eager_load_joined():
    with Session(engine) as s:
        # 1. Один запрос с LEFT OUTER JOIN: пилоты и их самолёты приходят вместе
        stmt = select(Pilot).options(joinedload(Pilot.aircrafts))
        # 2. .unique() склеивает дубли пилотов обратно в одного
        pilots = s.scalars(stmt).unique().all()
        for p in pilots:
            _ = p.aircrafts                         # данные уже в памяти -> новых запросов нет
        return len(pilots)

# --- Шаг 5: считаем запросы всеми тремя способами ---
_, n_joined = count_queries(eager_load_joined)
print("ленивая (N+1)  :", n_lazy,   "запросов")    # 6 = 1 + 5
print("selectinload   :", n_eager,  "запроса")     # 2
print("joinedload     :", n_joined, "запрос")      # 1

# ПОЧЕМУ selectinload РЕКОМЕНДУЮТ ЧАЩЕ:
# 1) joinedload повторяет ВСЕ колонки родителя в каждой строке JOIN -> лишний трафик на «широких» таблицах.
# 2) При нескольких коллекциях сразу строки перемножаются (m x n), результат раздувается.
# 3) С LIMIT по родителям joinedload приходится оборачивать в подзапрос, запрос усложняется.
# 4) selectinload шлёт простой второй запрос "... WHERE parent_id IN (...)", данные не дублируются.
# joinedload хорош для связей «многие-к-одному» и когда критично именно число запросов.

Без unique(): The unique() method must be invoked on this Result, as it contains results that include joined eager ...
ленивая (N+1)  : 6 запросов
selectinload   : 2 запроса
joinedload     : 1 запрос
